In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# =========================
# Paramètres repris du code
# =========================

TIME_SIMU = 300.0       # minutes
dt = 0.001              # minutes
n_steps = int(TIME_SIMU / dt)

J = 0.4
k_PDE = 1e5

F1_base = 2.0
F2_base = 0.04
N_HILL = 4
K_h = 3e-8

q_s = 6e-6
k_t = 0.8
b_max = q_s / k_t

K_relay = 5e-5

PDE_rate = 0.1
PDE_threshold = 5e-8
hill_n_PDE = 2
PDE_decay = 0.4
PDE_inhibition_threshold = 3e-6

# Cellule pionnière
f0 = 0.05

# =========================
# Conditions initiales
# =========================

c = 0.0       # cAMP extracellulaire
b = 0.0       # cAMP intracellulaire
r_T = 1.0     # récepteurs actifs
PDE = 0.0

times = np.zeros(n_steps)
c_trace = np.zeros(n_steps)
b_trace = np.zeros(n_steps)
r_trace = np.zeros(n_steps)
pde_trace = np.zeros(n_steps)

# =========================
# Simulation 0D
# =========================

for i in range(n_steps):
    t = i * dt

    c_pos = max(c, 0.0)

    hill_c = c_pos**N_HILL / (K_h**N_HILL + c_pos**N_HILL + 1e-60)

    f1_eff = F1_base * hill_c

    F_val = r_T * (f0 + (1.0 - f0) * hill_c)

    inhibition = 1.0 / (1.0 + (PDE / PDE_inhibition_threshold)**2)

    camp_prod = K_relay * (b / b_max) * inhibition
    camp_deg = J * c + k_PDE * PDE * c

    pde_prod = PDE_rate * c_pos**hill_n_PDE / (
        PDE_threshold**hill_n_PDE + c_pos**hill_n_PDE + 1e-60
    )

    dc = camp_prod - camp_deg
    db = q_s * F_val - k_t * b
    dr = -f1_eff * r_T + F2_base * (1.0 - r_T)
    dPDE = pde_prod - PDE_decay * PDE

    c += dc * dt
    b += db * dt
    r_T += dr * dt
    PDE += dPDE * dt

    c = max(c, 0.0)
    b = max(b, 0.0)
    r_T = min(max(r_T, 0.0), 1.0)
    PDE = max(PDE, 0.0)

    times[i] = t
    c_trace[i] = c
    b_trace[i] = b
    r_trace[i] = r_T
    pde_trace[i] = PDE

# =========================
# Affichage
# =========================

fig, axes = plt.subplots(4, 1, figsize=(10, 9), sharex=True)

axes[0].plot(times, c_trace * 1e9)
axes[0].set_ylabel("cAMP ext. (nM)")

axes[1].plot(times, b_trace / b_max)
axes[1].set_ylabel("b / bmax")

axes[2].plot(times, r_trace)
axes[2].set_ylabel("r_T")

axes[3].plot(times, pde_trace)
axes[3].set_ylabel("PDE")
axes[3].set_xlabel("Temps (min)")

plt.tight_layout()
plt.show()